In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# create our numpy random number generator
rng = np.random.default_rng(42)

# Review: Simulating Sampling Distributions

As we've seen previously, simulating a sampling distribution requires a simulator for an underlying distribution, which we run many times, computing some statistic such as the mean for each sample

In [ ]:
def simulate_sampling_distribution(
    underlying_distribution_simulator, sample_statistic, n_simulations=1000, sample_size=100
):
    samples = [underlying_distribution_simulator(sample_size) for _ in range(n_simulations)]
    return pd.DataFrame(
        {
            "statistic": [sample_statistic(sample) for sample in samples],
        }
    )

# Simulating Bootstrap Samples

Suppose that, instead of having a simulator for the underlying distribution that generated data, all we have is a sample of data. If we think that data was a random sample from the population, we can generate another random sample from the population by taking a random sample from the original sample. It's called a resample.

To build intuitions, suppose we have a standard deck of 52 playing cards. 
- If we want to deal a random 5-card poker hand, we could shuffle the deck well and take the top 5 cards.
- Alternatively, we could shuffle the deck and keep only the top half, 26 cards, and call it our original sample. Without looking at those 26 cards, we could shuffle them and deal the top five.
  - That subsample of five would be just as "random" as taking the top five cards from the full deck.

Now, if we want to generate many random 5-card poker hands that are just like we would have gotten from dealing from the 52-card deck, we can deal from the set of 26 many times.
-  We will use a process called sampling **with replacement**.
  - After we deal one hand of five cards, we put those cards back in and reshuffle before dealing another hand (not like playing an actual game of poker!)
  - Actually, we go one step farther. After we deal *each card*, we put it back and reshuffle the 26 cards.
- You may be bothered by this process, for a couple reasons:
    - You might get the Ace of Spades more than once in a single five-card hand, because of the "with replacement" process.
    - If the King of Hearts is not in the original sample of 26 cards, it will *never* appear in any of the five-card hands that you simulate.
- But it turns out this sampling with replacement process works pretty well for statistical inference about the sampling distribution in many settings.
    - Usually, we don't care about duplicating particular items or making sure that every item in the population has an equal chance of appearing in the samples.
    - Instead, we care about the samples having the same statistical properties that samples drawn from the population would have:
        - same mean.
        - independence from other samples.
    - In most cases, sampling with replacement will do that well. In particular, the "with replacement" approach does a better job of assuring independence between samples.

Because sampling with replacement is useful statistically, the software libraries we use have that capability built in!

#### `rng.choice()`
In particular, `rng.choice()` can be used to sample from an existing dataframe. 
- It returns a new dataframe with a specified number of rows selected from the existing dataframe
- If `replace=True` is specified, after each row is selected, it is replaced and is possibly available again as the next choice

In [ ]:
def bootstrap_resample(arr, n_rows):
    return rng.choice(arr, n_rows, replace=True)

Let's try it with half a deck of cards (represented as the numbers 1-52)

In [ ]:
random_half_deck = rng.choice(np.arange(1,53), 26, replace=False)
random_half_deck

In [ ]:
random_half_deck.mean()

In [ ]:
for _ in range(5):
    print(bootstrap_resample(random_half_deck, 5))

## Using Bootstrap Samples to Simulate the Sampling Distribution of the Mean


In [ ]:
sampling_dist_1 = simulate_sampling_distribution(
    underlying_distribution_simulator = lambda n: bootstrap_resample(random_half_deck, n),
    sample_statistic = np.mean,
    n_simulations=10000,
    sample_size=5
)
    

In [ ]:
sampling_dist_1.describe()

Let's see if the sampling distribution is very different than we would have gotten from sampling from a full deck

In [ ]:
full_deck = np.arange(1, 53)
sampling_dist_2 = simulate_sampling_distribution(
    underlying_distribution_simulator = lambda n: bootstrap_resample(full_deck, n),
    sample_statistic = np.mean,
    n_simulations=10000,
    sample_size=5
)
sampling_dist_2.describe()

In [ ]:
# plot the histogram of a dataset generated from a continuous distribution, by binning the data
def plot_continuous(data, ax, colname="statistic", num_bins=100, min_x=None, max_x=None):
    sns.histplot(data[colname], ax=ax, stat="probability")

    # Set the range of the x-axis if min_x and max_x are provided
    if min_x is not None and max_x is not None:
        ax.set_xlim(min_x, max_x)

    ax.set_ylabel("Probability")

In [ ]:
# use plot_continuous to plot both sampling distributions side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
plot_continuous(sampling_dist_1, ax2, min_x=0, max_x=53)
ax2.set_title("Sampling Distribution: Bootstrap Samples from a Half Deck")
plot_continuous(sampling_dist_2, ax1, min_x=0, max_x=53)
ax1.set_title("Sampling Distribution: Drawing from a Full Deck")
plt.tight_layout()
plt.show()

The sampling distributions came out pretty close, vindicating our approach of bootstrap sampling. 

It's a little risky, though.
- Imagine we had gotten unlucky and our initial half-deck was quite unrepresentative.

In [ ]:
weird_half_deck = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 52])
sampling_dist_3 = simulate_sampling_distribution(
    underlying_distribution_simulator = lambda n: bootstrap_resample(weird_half_deck, n),
    sample_statistic = np.mean,
    n_simulations=10000,
    sample_size=5
)
sampling_dist_3.describe()

In [ ]:
# use plot_continuous to plot all three sampling distributions side by side
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 6))
plot_continuous(sampling_dist_1, ax2, min_x=0, max_x=53)
ax2.set_title("Bootstrap Samples from a Random Half Deck")
plot_continuous(sampling_dist_2, ax1, min_x=0, max_x=53)
ax1.set_title("Drawing from a Full Deck")
plot_continuous(sampling_dist_3, ax3, min_x=0, max_x=53)
ax3.set_title("Bootstrap Samples from a Weird Half Deck")
plt.tight_layout()
plt.show()